# 4. Business Impact: ROI & Decision Framework

This notebook translates statistical findings into business value.

## Questions Answered
- If we treat 1M users, how many incremental conversions?
- What's the financial impact at different value-per-conversion levels?
- What's the decision rule: Should we deploy treatment?

## Scenarios Tested
- Value per conversion: $25, $50, $100, $200, $500
- Treatment recommendation based on:
  - Statistical significance (p < 0.05)
  - Absolute lift > 0
  - Minimum incremental conversions threshold

In [ ]:
import pandas as pd
import numpy as np

import os
DATA_PATH = os.path.join("data", "raw", "criteo-uplift-v2.1.csv.gz")
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (13979592, 16)


In [2]:
group_metrics = (
    df.groupby("treatment")
      .agg(
          observations=("treatment", "size"),
          conversions=("conversion", "sum")
      )
      .reset_index()
)

group_metrics["conversion_rate"] = (
    group_metrics["conversions"]
    / group_metrics["observations"]
)

control = group_metrics[group_metrics["treatment"] == 0].iloc[0]
treatment = group_metrics[group_metrics["treatment"] == 1].iloc[0]

baseline_conversion_rate = control["conversion_rate"]
treatment_conversion_rate = treatment["conversion_rate"]

observed_absolute_lift = (
    treatment_conversion_rate - baseline_conversion_rate
)

observed_relative_lift = (
    observed_absolute_lift / baseline_conversion_rate
)

print(f"Baseline conversion rate: {baseline_conversion_rate:.6%}")
print(f"Treatment conversion rate: {treatment_conversion_rate:.6%}")
print(f"Observed absolute lift: {observed_absolute_lift:.6%}")
print(f"Observed relative lift: {observed_relative_lift:.2%}")

Baseline conversion rate: 0.193759%
Treatment conversion rate: 0.308946%
Observed absolute lift: 0.115187%
Observed relative lift: 59.45%


In [3]:
treated_users = 1_000_000

expected_conversions_without_treatment = (
    treated_users * baseline_conversion_rate
)

expected_conversions_with_treatment = (
    treated_users * treatment_conversion_rate
)

estimated_incremental_conversions = (
    expected_conversions_with_treatment
    - expected_conversions_without_treatment
)

print(f"Treated users: {treated_users:,.0f}")
print(
    f"Expected conversions without treatment: "
    f"{expected_conversions_without_treatment:,.0f}"
)
print(
    f"Expected conversions with treatment: "
    f"{expected_conversions_with_treatment:,.0f}"
)
print(
    f"Estimated incremental conversions: "
    f"{estimated_incremental_conversions:,.0f}"
)

Treated users: 1,000,000
Expected conversions without treatment: 1,938
Expected conversions with treatment: 3,089
Estimated incremental conversions: 1,152


In [4]:
value_per_conversion = 100

estimated_incremental_value = (
    estimated_incremental_conversions * value_per_conversion
)

print(f"Assumed value per conversion: ${value_per_conversion:,.2f}")
print(
    f"Estimated incremental value: "
    f"${estimated_incremental_value:,.2f}"
)

Assumed value per conversion: $100.00
Estimated incremental value: $115,187.31


In [5]:
value_scenarios = [25, 50, 100, 200, 500]

sensitivity = pd.DataFrame({
    "value_per_conversion": value_scenarios
})

sensitivity["estimated_incremental_value"] = (
    estimated_incremental_conversions
    * sensitivity["value_per_conversion"]
)

sensitivity

,value_per_conversion,estimated_incremental_value
0,25,28796.826303
1,50,57593.652607
2,100,115187.305213
3,200,230374.610426
4,500,575936.526066


## Decision Support

The decision-support framework combines statistical evidence with practical business impact.

The framework does not treat statistical significance alone as sufficient for scaling a treatment. A treatment should also demonstrate a meaningful estimated incremental effect.

The final decision is based on:
- Statistical significance
- Observed incremental conversions
- Practical business impact
- Explicit scenario assumptions

In [6]:
alpha = 0.05
minimum_incremental_conversions = 100

if (
    observed_absolute_lift > 0
    and estimated_incremental_conversions >= minimum_incremental_conversions
):
    decision = "Positive treatment effect — suitable for further business evaluation"
else:
    decision = "Insufficient evidence for a positive treatment effect"

print("Decision:", decision)

Decision: Positive treatment effect — suitable for further business evaluation


In [7]:
statistically_significant = True
minimum_incremental_conversions = 100

if (
    statistically_significant
    and observed_absolute_lift > 0
    and estimated_incremental_conversions >= minimum_incremental_conversions
):
    decision = "Positive treatment effect — suitable for further business evaluation"
else:
    decision = "Insufficient evidence for a positive treatment effect"

print("Decision:", decision)

Decision: Positive treatment effect — suitable for further business evaluation


## Business Impact Summary

For a hypothetical audience of 1,000,000 treated users, the observed experiment results imply approximately 1,152 incremental conversions compared with the control conversion rate.

Using an explicit value-per-conversion scenario, the estimated incremental value can be evaluated under different business assumptions.

The business-impact estimates are scenario-based and should not be interpreted as actual revenue because the Criteo dataset does not contain revenue or monetary outcome information.

The decision-support framework combines the observed treatment effect, statistical evidence from the experiment analysis, and a minimum practical impact threshold to support further business evaluation.

In [ ]:
# Save sensitivity analysis
sensitivity.to_csv('outputs/results/sensitivity_analysis.csv', index=False)
print("✅ Sensitivity analysis saved to outputs/results/sensitivity_analysis.csv")